In [1]:
import pandas as pd
from filters import ArticleFilterer

In [2]:
projects = pd.read_json("../src/data/projects.jsonl", lines=True)
projects = projects[projects["total_space_sqft"] >= 100_000]  # greater than 100k sqft = hyperscale
print(len(projects), "total hyperscale projects in us")

# get candidate links
articles = pd.read_json("../src/data/articles.jsonl", lines=True)[["rss_url", "text", "status"]]
articles = articles.query("status == 'ok'")

headlines = pd.read_json("../src/data/headlines.jsonl", lines=True)[
    ["slug", "title", "rss_url", "published", "source", "status"]]
headlines["published"] = pd.to_datetime(headlines["published"])
headlines = headlines.query("status == 'ok'")

articles = headlines.merge(articles, how="left", on="rss_url")
articles = articles.dropna(subset="text").reset_index(drop=True)

print(len(articles), "candidate links")
print(articles["slug"].nunique(), "hyperscale projects with coverage")

# filter articles
merged = articles.merge(projects, on="slug", how="left")

filterer = ArticleFilterer()
filtered = filterer.filter(merged)
print(len(filtered), "articles after filtering")

783 total hyperscale projects in us
8117 candidate links
1296 hyperscale projects with coverage
4048 articles after filtering
